# Screw Detection Visualization Demo

Dieses Notebook zeigt Bounding Boxes direkt aus Repository-Daten an.

Es erwartet bevorzugt Daten unter `text_bilder/` und faellt fuer die aktuelle Testumgebung auf `test_bilder/` zurueck.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import pandas as pd
from PIL import Image


In [ ]:
CANDIDATE_DATASETS = [Path('text_bilder'), Path('test_bilder')]

dataset_root = next((path for path in CANDIDATE_DATASETS if path.exists()), None)
if dataset_root is None:
    raise FileNotFoundError('Kein Datensatz gefunden. Erwartet wird text_bilder/ oder test_bilder/.')

images_dir = dataset_root / 'bilder'
boxed_images_dir = dataset_root / 'bilder_mit_boxen'
labels_dir = dataset_root / 'bounding_boxen'

print('Datensatz:', dataset_root)
print('Bilder:', images_dir.exists())
print('Bounding Boxes:', labels_dir.exists())
print('Vergleichsbilder mit Boxen:', boxed_images_dir.exists())


In [ ]:
def list_image_files(folder: Path):
    patterns = ('*.jpg', '*.jpeg', '*.png', '*.webp')
    files = []
    for pattern in patterns:
        files.extend(folder.glob(pattern))
    return sorted(files)


def parse_yolo_boxes(label_path: Path):
    rows = []
    if not label_path.exists():
        return rows

    for line_no, raw_line in enumerate(label_path.read_text().splitlines(), start=1):
        line = raw_line.strip()
        if not line:
            continue

        parts = line.split()
        if len(parts) != 5:
            raise ValueError(f'Ungueltiges Label-Format in {label_path} Zeile {line_no}: {raw_line!r}')

        class_id, x_center, y_center, width, height = parts
        rows.append({
            'class_id': int(float(class_id)),
            'x_center_norm': float(x_center),
            'y_center_norm': float(y_center),
            'width_norm': float(width),
            'height_norm': float(height),
        })

    return rows


def find_matching_file(folder: Path, stem: str):
    if not folder.exists():
        return None
    for extension in ('.jpg', '.jpeg', '.png', '.webp'):
        candidate = folder / f'{stem}{extension}'
        if candidate.exists():
            return candidate
    return None


def build_index(dataset_root: Path):
    rows = []
    for image_path in list_image_files(dataset_root / 'bilder'):
        label_path = dataset_root / 'bounding_boxen' / f'{image_path.stem}.txt'
        boxed_image_path = find_matching_file(dataset_root / 'bilder_mit_boxen', image_path.stem)
        boxes = parse_yolo_boxes(label_path)
        rows.append({
            'image_name': image_path.name,
            'stem': image_path.stem,
            'image_path': str(image_path),
            'label_path': str(label_path) if label_path.exists() else '',
            'boxed_image_path': str(boxed_image_path) if boxed_image_path else '',
            'box_count': len(boxes),
        })
    return pd.DataFrame(rows)


def draw_boxes(ax, image_width: int, image_height: int, boxes):
    for box in boxes:
        width_px = box['width_norm'] * image_width
        height_px = box['height_norm'] * image_height
        x_min = (box['x_center_norm'] * image_width) - (width_px / 2)
        y_min = (box['y_center_norm'] * image_height) - (height_px / 2)

        rect = Rectangle(
            (x_min, y_min),
            width_px,
            height_px,
            linewidth=2,
            edgecolor='lime',
            facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(
            x_min,
            max(0, y_min - 4),
            f"class {box['class_id']}",
            color='black',
            fontsize=8,
            bbox={'facecolor': 'lime', 'edgecolor': 'none', 'pad': 1}
        )


def show_example(stem: str):
    image_path = find_matching_file(images_dir, stem)
    if image_path is None:
        raise FileNotFoundError(f'Kein Bild fuer {stem!r} gefunden.')

    label_path = labels_dir / f'{stem}.txt'
    boxed_image_path = find_matching_file(boxed_images_dir, stem)
    boxes = parse_yolo_boxes(label_path)

    image = Image.open(image_path)
    width, height = image.size

    has_reference = boxed_image_path is not None
    fig_cols = 3 if has_reference else 2
    fig, axes = plt.subplots(1, fig_cols, figsize=(6 * fig_cols, 6))

    axes[0].imshow(image)
    axes[0].set_title(f'Original: {image_path.name}')
    axes[0].axis('off')

    axes[1].imshow(image)
    draw_boxes(axes[1], width, height, boxes)
    axes[1].set_title(f'Rendered boxes: {len(boxes)}')
    axes[1].axis('off')

    if has_reference:
        reference_image = Image.open(boxed_image_path)
        axes[2].imshow(reference_image)
        axes[2].set_title('Reference image with boxes')
        axes[2].axis('off')

    plt.tight_layout()
    plt.show()

    return pd.DataFrame(boxes)


## Verfuegbare Bilder

Die folgende Tabelle zeigt alle gefundenen Bilder und wie viele Bounding Boxes aus den `.txt`-Dateien gelesen wurden.


In [ ]:
index_df = build_index(dataset_root)
if index_df.empty:
    raise ValueError(f'Keine Bilder in {images_dir} gefunden.')

index_df

## Erstes Beispiel rendern

Hier wird das erste gefundene Bild automatisch angezeigt. Fuer ein anderes Bild kannst du unten einfach einen anderen `stem` einsetzen.


In [ ]:
default_stem = index_df.iloc[0]['stem']
boxes_df = show_example(default_stem)
boxes_df

In [ ]:
# Beispiel fuer manuelle Auswahl:
# show_example('6afb35f6-cc0f-4713-902d-60ac24361256')
